# Hawkes/SVMHJD Dataset Benchmark

This notebook analyses the synthetic stochastic-volatility marked Hawkes jump-diffusion (SVMHJD) dataset and compares the exact Ogata event backend against the fixed-grid approximation.

A compact SVMHJD path can be written as

$$
S_{t_i} = S_{t_{i-1}} \exp(r_i), \qquad
r_i = \mu\Delta t + \int_{t_{i-1}}^{t_i} \sigma_u\,dW_u + \sum_{t_{i-1} < \tau_j \leq t_i} Y_j .
$$

The marked Hawkes jump intensity is

$$
\lambda_t = \lambda_0 + \sum_{\tau_j < t}\left(\alpha + \eta |Y_j|\right)\exp[-\beta(t - \tau_j)],
$$

so the implementation includes mark-dependent excitation through the $\eta |Y_j|$ term. Jump magnitudes also excite volatility through

$$
\sigma_t = \min\left(\sigma_{\max}, \sigma_0 + \sum_{\tau_j < t} \gamma |Y_j|\exp[-\kappa(t - \tau_j)]\right).
$$

The baseline Hawkes stability condition used by the simulator is excitation / decay $= \alpha / \beta < 1$; the diagnostic metadata reports this branching-ratio proxy. The intensity is additionally capped in the implementation, and mark excitation is used for stress-test dynamics.

This is a synthetic benchmark, not a no-arbitrage pricing model. The paths are intended for generative-modelling diagnostics, tail-risk stress tests, and leakage checks rather than calibrated trading or derivative-pricing use.

Two backends are exposed through `HawkesJumpDataset`. The fixed-grid backend samples arrivals from a discrete-time Poisson approximation at each observation step. The Ogata backend samples continuous-time marked Hawkes arrivals by thinning and then projects events, intensities, volatility, and prices onto the same observation grid. Ogata is therefore the preferred backend for public event-timing analysis, while fixed-grid remains useful for fast smoke checks.


## 1. Imports and parameters

The default run is a lightweight smoke analysis. Set `RUN_FULL=True` locally for the larger Monte Carlo sample; no model training is invoked by this notebook.


In [ ]:
from __future__ import annotations

import os
import sys
import time
from collections.abc import Mapping
from pathlib import Path
from typing import Any

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-time-causal-vae")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from time_causal_vae.data.hawkes_jump import HawkesJumpDataset
from time_causal_vae.evaluation.jump_diagnostics import jump_diagnostic_summary
from time_causal_vae.evaluation.market_diagnostics import (
    compute_log_returns,
    distribution_summary,
    maximum_drawdown,
)

try:
    from time_causal_vae.evaluation.market_diagnostics import market_style_summary
except ImportError:  # pragma: no cover - public notebook fallback for older checkouts.
    market_style_summary = None

try:
    from scripts.plot_hawkes_jump_dataset import inter_arrival_gaps
except Exception:  # pragma: no cover - keep notebook usable when scripts are not importable.

    def inter_arrival_gaps(jump_indicators: torch.Tensor) -> torch.Tensor:
        squeezed = to_2d_tensor(jump_indicators).bool()  # type: ignore
        gaps: list[float] = []
        for path_indicators in squeezed:
            positions = torch.nonzero(path_indicators, as_tuple=False).flatten().float()
            if positions.numel() > 1:
                gaps.extend(positions.diff().tolist())
        return torch.tensor(gaps, dtype=torch.float32)


RUN_SMOKE = True
RUN_FULL = False
N_SAMPLES_SMOKE = 256
N_SAMPLES_FULL = 4096
SIMULATION_SCHEME = "ogata"
COMPARE_FIXED_GRID = True

N_TIMESTEPS = 60
SEED = 99
VOLATILITY_EXCITATION = True
DT = 1.0 / 60.0

if SIMULATION_SCHEME not in {"ogata", "fixed_grid"}:
    raise ValueError("SIMULATION_SCHEME must be 'ogata' or 'fixed_grid'.")

N_SAMPLES = N_SAMPLES_FULL if RUN_FULL else N_SAMPLES_SMOKE
RUN_MODE = "full" if RUN_FULL else "smoke"

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

display(
    pd.DataFrame([
        {"parameter": "RUN_SMOKE", "value": RUN_SMOKE},
        {"parameter": "RUN_FULL", "value": RUN_FULL},
        {"parameter": "N_SAMPLES_SMOKE", "value": N_SAMPLES_SMOKE},
        {"parameter": "N_SAMPLES_FULL", "value": N_SAMPLES_FULL},
        {"parameter": "N_SAMPLES_EFFECTIVE", "value": N_SAMPLES},
        {"parameter": "N_TIMESTEPS", "value": N_TIMESTEPS},
        {"parameter": "SIMULATION_SCHEME", "value": SIMULATION_SCHEME},
        {"parameter": "COMPARE_FIXED_GRID", "value": COMPARE_FIXED_GRID},
        {"parameter": "SEED", "value": SEED},
        {"parameter": "VOLATILITY_EXCITATION", "value": VOLATILITY_EXCITATION},
    ])
)

In [ ]:
def to_2d_tensor(values: torch.Tensor) -> torch.Tensor:
    """Return a CPU float tensor with shape [batch, time]."""
    squeezed = values.detach().cpu()
    if squeezed.ndim == 3 and squeezed.shape[-1] == 1:
        squeezed = squeezed[..., 0]
    if squeezed.ndim != 2:
        raise ValueError(f"Expected [batch, time] or [batch, time, 1], got {tuple(values.shape)}.")
    return squeezed.float()


def to_numpy_2d(values: torch.Tensor) -> np.ndarray:
    """Return a NumPy array with shape [batch, time]."""
    return to_2d_tensor(values).numpy()


def available_tensor(dataset: HawkesJumpDataset, name: str) -> torch.Tensor | None:
    """Read an optional simulator tensor from a dataset."""
    value = getattr(dataset, name, None)
    return value if isinstance(value, torch.Tensor) else None


def safe_market_style_summary(paths: torch.Tensor) -> dict[str, Any]:
    """Use the repository market summary when available, otherwise compute basics."""
    if market_style_summary is not None:
        return market_style_summary(paths)
    returns = compute_log_returns(paths)
    return {
        "path_shape": list(paths.shape),
        "returns": distribution_summary(returns),
        "maximum_drawdown": distribution_summary(maximum_drawdown(paths)),
    }


def simulate_and_summarise(scheme: str) -> tuple[HawkesJumpDataset, dict[str, Any]]:
    """Generate one Hawkes/SVMHJD dataset and diagnostics in memory."""
    started = time.perf_counter()
    dataset = HawkesJumpDataset(
        N_SAMPLES,
        N_TIMESTEPS,
        seed=SEED,
        simulation_scheme=scheme,
        volatility_excitation=VOLATILITY_EXCITATION,
        dt=DT,
    )
    runtime_seconds = time.perf_counter() - started
    jump_summary = jump_diagnostic_summary(
        dataset.prices,
        jump_indicators=available_tensor(dataset, "jump_indicators"),
        jump_counts=available_tensor(dataset, "jump_counts"),
        jump_sizes=available_tensor(dataset, "jump_sizes"),
    )
    summary = {
        "config": {
            "run_mode": RUN_MODE,
            "n_samples": N_SAMPLES,
            "n_timesteps": N_TIMESTEPS,
            "seed": SEED,
            "simulation_scheme": scheme,
            "volatility_excitation": VOLATILITY_EXCITATION,
            "dt": DT,
        },
        "runtime_seconds": runtime_seconds,
        "tensor_shapes": {
            "prices": list(dataset.prices.shape),
            "log_returns": list(dataset.log_returns.shape),
            "jump_indicators": list(dataset.jump_indicators.shape),
            "jump_counts": list(dataset.jump_counts.shape),
            "jump_sizes": list(dataset.jump_sizes.shape),
            "intensities": list(dataset.intensities.shape),
            "volatilities": list(dataset.volatilities.shape),
        },
        "dataset_metadata": dict(dataset.metadata),
        "jump_summary": jump_summary,
        "market_summary": safe_market_style_summary(dataset.prices),
        "checks": {
            "prices_positive": bool((dataset.prices > 0.0).all().item()),
            "prices_finite": bool(torch.isfinite(dataset.prices).all().item()),
            "log_returns_finite": bool(torch.isfinite(dataset.log_returns).all().item()),
            "has_jumps": bool((dataset.jump_counts.sum() > 0).item()),
            "branching_ratio_below_one": bool(dataset.metadata["branching_ratio_proxy"] < 1.0),
        },
    }
    return dataset, summary


def comparison_row(summary: Mapping[str, Any]) -> dict[str, Any]:
    """Extract a compact Monte Carlo comparison row."""
    metadata = summary["dataset_metadata"]
    jumps = summary["jump_summary"]
    market = summary["market_summary"]
    drawdown = market.get("maximum_drawdown", {})
    var_es = jumps["var_es"]
    clustering = jumps["clustering"]
    returns = market.get("returns", {})
    return {
        "scheme": summary["config"]["simulation_scheme"],
        "runtime_s": summary["runtime_seconds"],
        "total_jumps": metadata["total_jumps"],
        "mean_jumps_per_path": metadata["mean_jump_count_per_path"],
        "paths_with_jumps": metadata["paths_with_jump_fraction"],
        "negative_jump_fraction": metadata["negative_jump_fraction"],
        "branching_ratio_proxy": metadata["branching_ratio_proxy"],
        "count_overdispersion": clustering["count_overdispersion"],
        "adjacent_jump_pairs": clustering["adjacent_jump_pair_count"],
        "return_mean": returns.get("mean", 0.0),
        "return_q001": returns.get("q001", 0.0),
        "return_q999": returns.get("q999", 0.0),
        "lower_tail_var_q01": var_es["lower_tail_var_q01"],
        "lower_tail_es_q01": var_es["lower_tail_es_q01"],
        "max_drawdown_mean": drawdown.get("mean", 0.0),
        "max_drawdown_q99": drawdown.get("q99", 0.0),
        "max_intensity_observed": metadata["max_intensity_observed"],
        "max_volatility_observed": metadata["max_volatility_observed"],
    }


def comparison_frame(summaries: Mapping[str, Mapping[str, Any]]) -> pd.DataFrame:
    """Build a rounded comparison frame for all generated schemes."""
    rows = [comparison_row(summary) for summary in summaries.values()]
    return pd.DataFrame(rows).round(6)


def tail_exceedance_frame(
    datasets: Mapping[str, HawkesJumpDataset],
    summaries: Mapping[str, Mapping[str, Any]],
    *,
    reference_scheme: str = "ogata",
) -> pd.DataFrame:
    """Compare VaR, ES, and exceedance rates against one reference threshold set."""
    reference = reference_scheme if reference_scheme in summaries else next(iter(summaries))
    thresholds = summaries[reference]["jump_summary"]["tail_thresholds"]
    rows: list[dict[str, float | int | str]] = []
    for scheme, dataset in datasets.items():
        returns = compute_log_returns(dataset.prices).flatten()
        var_es = summaries[scheme]["jump_summary"]["var_es"]
        row: dict[str, float | int | str] = {
            "scheme": scheme,
            "threshold_source": reference,
            "return_count": int(returns.numel()),
            "lower_tail_var_q01": var_es["lower_tail_var_q01"],
            "lower_tail_es_q01": var_es["lower_tail_es_q01"],
            "lower_tail_var_q05": var_es["lower_tail_var_q05"],
            "lower_tail_es_q05": var_es["lower_tail_es_q05"],
        }
        for name, threshold in thresholds.items():
            digits = "".join(character for character in name if character.isdigit())
            probability = float("0." + digits) if digits else 0.0
            if probability < 0.5:
                mask = returns <= float(threshold)
                prefix = "below"
            else:
                mask = returns >= float(threshold)
                prefix = "above"
            row[f"{prefix}_{name}_count"] = int(mask.sum().item())
            row[f"{prefix}_{name}_fraction"] = float(mask.float().mean().item())
        rows.append(row)
    return pd.DataFrame(rows).round(6)


def nonzero_jump_sizes(dataset: HawkesJumpDataset) -> np.ndarray:
    """Return non-zero aggregate jump sizes for one dataset."""
    sizes = to_2d_tensor(dataset.jump_sizes).reshape(-1)
    sizes = sizes[sizes.abs() > 0.0]
    return sizes.numpy()


def jump_counts_per_path(dataset: HawkesJumpDataset) -> np.ndarray:
    """Return per-path total jump counts."""
    return to_2d_tensor(dataset.jump_counts).sum(dim=1).numpy()


def ordered_items(datasets: Mapping[str, HawkesJumpDataset]) -> list[tuple[str, HawkesJumpDataset]]:
    """Use stable plotting order."""
    order = [scheme for scheme in (SIMULATION_SCHEME, "ogata", "fixed_grid") if scheme in datasets]
    seen: set[str] = set()
    result: list[tuple[str, HawkesJumpDataset]] = []
    for scheme in order:
        if scheme in seen:
            continue
        seen.add(scheme)
        result.append((scheme, datasets[scheme]))
    return result

In [ ]:
def plot_sample_paths(
    datasets: Mapping[str, HawkesJumpDataset],
    *,
    tensor_name: str,
    title: str,
    ylabel: str,
    max_paths: int = 18,
) -> plt.Figure:
    """Plot a deterministic subset of path trajectories, following the script style."""
    items = ordered_items(datasets)
    fig, axes = plt.subplots(1, len(items), figsize=(6.2 * len(items), 4.0), squeeze=False)
    for ax, (scheme, dataset) in zip(axes[0], items, strict=True):
        paths = to_numpy_2d(getattr(dataset, tensor_name))
        for index in range(min(paths.shape[0], max_paths)):
            ax.plot(paths[index], linewidth=1.0, alpha=0.68)
        ax.set_title(f"{title}: {scheme}")
        ax.set_xlabel("Time step")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)
    fig.tight_layout()
    return fig


def plot_jump_raster(
    datasets: Mapping[str, HawkesJumpDataset],
    *,
    max_paths: int = 256,
) -> plt.Figure:
    """Plot jump indicators as path-by-time rasters."""
    items = ordered_items(datasets)
    fig, axes = plt.subplots(1, len(items), figsize=(6.2 * len(items), 4.2), squeeze=False)
    for ax, (scheme, dataset) in zip(axes[0], items, strict=True):
        indicators = to_numpy_2d(dataset.jump_indicators).astype(float)
        ax.imshow(indicators[:max_paths], aspect="auto", interpolation="nearest", cmap="Greys")
        ax.set_title(f"Jump raster: {scheme}")
        ax.set_xlabel("Time step")
        ax.set_ylabel("Path index")
    fig.tight_layout()
    return fig


def plot_state_trajectories(
    datasets: Mapping[str, HawkesJumpDataset],
    *,
    max_paths: int = 18,
) -> plt.Figure:
    """Plot Hawkes intensity and jump-excited volatility trajectories."""
    items = ordered_items(datasets)
    fig, axes = plt.subplots(2, len(items), figsize=(6.2 * len(items), 7.2), squeeze=False)
    for column, (scheme, dataset) in enumerate(items):
        for row, (tensor_name, ylabel) in enumerate((
            ("intensities", "Intensity"),
            ("volatilities", "Volatility"),
        )):
            ax = axes[row, column]
            values = to_numpy_2d(getattr(dataset, tensor_name))
            for index in range(min(values.shape[0], max_paths)):
                ax.plot(values[index], linewidth=1.0, alpha=0.68)
            ax.set_title(f"{ylabel}: {scheme}")
            ax.set_xlabel("Time step")
            ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.25)
    fig.tight_layout()
    return fig


def plot_jump_count_histogram(datasets: Mapping[str, HawkesJumpDataset]) -> plt.Figure:
    """Plot per-path jump-count histograms."""
    items = ordered_items(datasets)
    max_count = max(int(jump_counts_per_path(dataset).max()) for _, dataset in items)
    bins = np.arange(0, max_count + 2) - 0.5
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    for scheme, dataset in items:
        ax.hist(
            jump_counts_per_path(dataset), bins=bins, alpha=0.58, label=scheme, edgecolor="black"
        )
    ax.set_title("Jump count distribution")
    ax.set_xlabel("Jumps per path")
    ax.set_ylabel("Path count")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    return fig


def plot_inter_arrival_histogram(datasets: Mapping[str, HawkesJumpDataset]) -> plt.Figure:
    """Plot within-path grid-step gaps between consecutive jump steps."""
    items = ordered_items(datasets)
    gap_arrays = []
    for scheme, dataset in items:
        gaps = inter_arrival_gaps(dataset.jump_indicators).detach().cpu().numpy()
        gap_arrays.append((scheme, gaps))
    max_gap = max((int(gaps.max()) for _, gaps in gap_arrays if gaps.size), default=1)
    bins = np.arange(1, max_gap + 2) - 0.5
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    for scheme, gaps in gap_arrays:
        if gaps.size:
            ax.hist(gaps, bins=bins, alpha=0.58, label=scheme, edgecolor="black")
        else:
            ax.text(0.5, 0.5, f"No repeated jumps for {scheme}", transform=ax.transAxes)
    ax.set_title("Inter-arrival time distribution")
    ax.set_xlabel("Grid steps between jumps")
    ax.set_ylabel("Count")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    return fig


def plot_jump_size_distribution(datasets: Mapping[str, HawkesJumpDataset]) -> plt.Figure:
    """Plot non-zero aggregate jump-size distributions."""
    items = ordered_items(datasets)
    size_arrays = [(scheme, nonzero_jump_sizes(dataset)) for scheme, dataset in items]
    all_sizes = (
        np.concatenate([sizes for _, sizes in size_arrays if sizes.size])
        if any(sizes.size for _, sizes in size_arrays)
        else np.array([], dtype=float)
    )
    bins = np.linspace(float(all_sizes.min()), float(all_sizes.max()), 50) if all_sizes.size else 20
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    for scheme, sizes in size_arrays:
        if sizes.size:
            ax.hist(sizes, bins=bins, alpha=0.58, label=scheme, edgecolor="black")
    ax.axvline(0.0, color="black", linewidth=1.0)
    ax.set_title("Jump size distribution")
    ax.set_xlabel("Aggregate jump log-return")
    ax.set_ylabel("Jump-step count")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    return fig

## 2. Generate Ogata dataset

The Ogata dataset is always generated because it is the reference backend for continuous-time Hawkes event timing.


In [ ]:
datasets: dict[str, HawkesJumpDataset] = {}
summaries: dict[str, dict[str, Any]] = {}

ogata_dataset, ogata_summary = simulate_and_summarise("ogata")
datasets["ogata"] = ogata_dataset
summaries["ogata"] = ogata_summary

display(pd.DataFrame([comparison_row(ogata_summary)]).round(6))
display(pd.DataFrame([ogata_summary["checks"]]))

## 3. Generate fixed-grid dataset

The fixed-grid dataset is generated only when `COMPARE_FIXED_GRID=True`. It uses the same sample size, grid length, seed, and volatility-excitation setting as the Ogata run.


In [ ]:
if COMPARE_FIXED_GRID:
    fixed_grid_dataset, fixed_grid_summary = simulate_and_summarise("fixed_grid")
    datasets["fixed_grid"] = fixed_grid_dataset
    summaries["fixed_grid"] = fixed_grid_summary
    display(pd.DataFrame([comparison_row(fixed_grid_summary)]).round(6))
    display(pd.DataFrame([fixed_grid_summary["checks"]]))
else:
    display(Markdown("Fixed-grid comparison disabled by `COMPARE_FIXED_GRID=False`."))

## 4. Compare runtime and MC statistics

This table compares wall-clock simulation time and Monte Carlo diagnostics across the generated backends.


In [ ]:
mc_comparison = comparison_frame(summaries)
display(mc_comparison)

## 5. Plot sample price paths


In [ ]:
fig = plot_sample_paths(datasets, tensor_name="prices", title="Sample price paths", ylabel="Price")
display(fig)
plt.close(fig)

## 6. Plot sample log-return paths


In [ ]:
fig = plot_sample_paths(
    datasets,
    tensor_name="log_returns",
    title="Sample log-return paths",
    ylabel="Log return",
)
display(fig)
plt.close(fig)

## 7. Jump indicator raster


In [ ]:
fig = plot_jump_raster(datasets)
display(fig)
plt.close(fig)

## 8. Intensity and volatility trajectories


In [ ]:
fig = plot_state_trajectories(datasets)
display(fig)
plt.close(fig)

## 9. Jump-count histogram


In [ ]:
fig = plot_jump_count_histogram(datasets)
display(fig)
plt.close(fig)

## 10. Inter-arrival histogram


In [ ]:
fig = plot_inter_arrival_histogram(datasets)
display(fig)
plt.close(fig)

## 11. Jump-size distribution


In [ ]:
fig = plot_jump_size_distribution(datasets)
display(fig)
plt.close(fig)

## 12. VaR/ES and tail exceedance table

The exceedance columns use the Ogata return-tail thresholds as the reference when that backend is available.


In [ ]:
risk_table = tail_exceedance_frame(datasets, summaries, reference_scheme="ogata")
display(risk_table)

## 13. Summary decision


In [ ]:
ogata_row = comparison_row(summaries["ogata"])
lines = [
    "### Decision",
    "",
    f"Ogata generated {int(ogata_row['total_jumps'])} jumps across {N_SAMPLES} paths "
    f"in {ogata_row['runtime_s']:.3f} seconds.",
    f"The reported Hawkes branching-ratio proxy is {ogata_row['branching_ratio_proxy']:.3f}, "
    "which satisfies the excitation / decay < 1 stability check.",
]

if "fixed_grid" in summaries:
    fixed_row = comparison_row(summaries["fixed_grid"])
    runtime_ratio = fixed_row["runtime_s"] / max(ogata_row["runtime_s"], 1e-12)
    lines.extend([
        "",
        f"The fixed-grid backend generated {int(fixed_row['total_jumps'])} jumps "
        f"in {fixed_row['runtime_s']:.3f} seconds, "
        f"or {runtime_ratio:.2f} times the Ogata runtime in this run.",
        f"Mean jumps per path were {ogata_row['mean_jumps_per_path']:.3f} for Ogata "
        f"and {fixed_row['mean_jumps_per_path']:.3f} for fixed-grid.",
        "",
        "Use Ogata for public dataset analysis and event-timing diagnostics because arrivals "
        "are simulated in continuous time before projection to the model grid. Use fixed-grid "
        "for quick smoke checks where the grid-level approximation is acceptable.",
    ])
else:
    lines.extend([
        "",
        "Fixed-grid comparison was disabled. Use Ogata for the public benchmark analysis by default.",
    ])

display(Markdown("\n".join(lines)))